# 프로젝트 4 - Weekend 2: 멀티모달 추출 + RAG 베이스라인

**이번 주말 목표** — 비정형 데이터의 **모든 모달리티**(PDF 표/이미지/차트, 오디오, 비디오)에서 정보를 끌어내고, **RAG 베이스라인**까지 완성한다. Weekend 3는 이 RAG를 Graph-RAG로 보강.

**학습 목표**:
1. `PyMuPDF`로 PDF 페이지 → 이미지, 표 영역 탐지·추출
2. `pandas` DataFrame과 Markdown 표로 변환·정제
3. GPT-4o Vision API로 이미지 캡셔닝과 차트 데이터 복원(Pydantic 구조화)
4. **Whisper**(LangChain `OpenAIWhisperParser`)로 오디오 → 텍스트, 청크 단위 Document
5. **OpenCV**로 비디오 프레임 샘플링, **ffmpeg**로 오디오 트랙 분리 후 transcribe
6. PDF·오디오·비디오를 단일 `Document` 스키마로 통합
7. **FAISS** 멀티모달 인덱싱 + `element_type` 가중치 검색
8. 멀티모달 컨텍스트 RAG 답변 + **LLM-as-Judge**로 충실도 검증


> 📦 실습 데이터:
> - PDF: `data/quarterly_report.pdf`, `data/research_paper.pdf` (Weekend 1과 동일)
> - 오디오: `data/audio/*.mp3` 3개 (회의 — 매출/제품/채용)
> - 비디오: `data/video/earnings_briefing.mp4` (실적 브리핑 60초)


In [ ]:
# 환경 설정 및 라이브러리 설치
# - ffmpeg는 시스템 패키지: 미설치 시 `sudo apt-get install -y ffmpeg` 또는 `brew install ffmpeg`
!pip install -q langchain langchain-openai langchain-community pymupdf pillow pandas pydantic python-dotenv \
    opencv-python openai faiss-cpu


In [ ]:
import os
import io
import base64
import time
import json
import subprocess
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

load_dotenv()

import fitz  # PyMuPDF
import cv2
import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 일부 문제는 Vision이 필요하지 않을 수 있음 — Vision 호출은 별도 시그니처로 명시
vision = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

DATA_DIR = Path("data")
PDF_REPORT = DATA_DIR / "quarterly_report.pdf"
PDF_PAPER = DATA_DIR / "research_paper.pdf"
AUDIO_DIR = DATA_DIR / "audio"
VIDEO_PATH = DATA_DIR / "video" / "earnings_briefing.mp4"

IMG_DIR = DATA_DIR / "_extracted"
FRAMES_DIR = DATA_DIR / "_frames"
AV_TMP_DIR = DATA_DIR / "_av_tmp"
for d in (IMG_DIR, FRAMES_DIR, AV_TMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert PDF_REPORT.exists(), f"❌ {PDF_REPORT} 파일이 없습니다."
assert AUDIO_DIR.exists() and any(AUDIO_DIR.glob("*.mp3")), f"❌ {AUDIO_DIR}/*.mp3 없음."
assert VIDEO_PATH.exists(), f"❌ {VIDEO_PATH} 없음."

audio_files = sorted(AUDIO_DIR.glob("*.mp3"))
print("✅ 환경 설정 완료")
print(f"📄 PDF: {PDF_REPORT.name}, {PDF_PAPER.name}")
print(f"🎙️  Audio: {len(audio_files)}개 — {[f.name for f in audio_files]}")
print(f"🎬 Video: {VIDEO_PATH.name} ({VIDEO_PATH.stat().st_size:,} bytes)")


---
## 📦 실습 데이터

| 카테고리 | 파일 | 내용 |
|----------|------|------|
| **PDF** | `quarterly_report.pdf` | 막대그래프, 라인차트, 실적표·KPI표 |
| **PDF** | `research_paper.pdf` | 파이프라인 그림, loss 그래프, 검색 성능표, 수식 |
| **Audio** | `audio/meeting_revenue.mp3` | 회의 발화 (매출) |
| **Audio** | `audio/meeting_product.mp3` | 회의 발화 (제품) |
| **Audio** | `audio/meeting_hiring.mp3` | 회의 발화 (채용) |
| **Video** | `video/earnings_briefing.mp4` | 실적 브리핑 영상 (PDF + 음성 합성) |

작업 디렉토리:
- `data/_extracted/` — PDF에서 뽑은 이미지
- `data/_frames/` — 비디오 프레임
- `data/_av_tmp/` — 비디오에서 분리한 오디오 등 임시


---
## 문제 1: PDF 페이지를 PNG 이미지로 렌더링

PyMuPDF로 PDF 각 페이지를 PNG로 변환하여 저장하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `render_pages_to_png(pdf_path: str, out_dir: str, dpi: int = 150) -> list[str]`
- 각 페이지를 `<out_dir>/<stem>_p<N>.png`로 저장
- 반환: 저장된 파일 경로 리스트

**평가기준:**
- 반환 길이 == 페이지 수
- 저장된 파일이 실제로 존재하고 PNG로 열림


In [ ]:
def render_pages_to_png(pdf_path: str, out_dir: str, dpi: int = 150) -> list[str]:
    """PDF 각 페이지를 PNG로 저장."""
    # ---- 여기에 코드 작성 ----
    # 1) fitz.open
    # 2) page.get_pixmap(dpi=dpi)
    # 3) pix.save(out_path)
    return []


# 테스트
paths = render_pages_to_png(str(PDF_REPORT), str(IMG_DIR), dpi=120)
print(f"렌더된 페이지: {len(paths)}")
for p in paths:
    sz = Path(p).stat().st_size
    print(f"  {Path(p).name}: {sz:,} bytes")


---
## 문제 2: PDF 안에 임베드된 이미지 추출

PyMuPDF로 페이지에 삽입된 raster 이미지(차트, 그림)를 별도 파일로 추출하세요.

**요구사항:**
- 함수 시그니처: `extract_embedded_images(pdf_path: str, out_dir: str) -> list[dict]`
- 각 dict: `{"page", "index", "path", "width", "height"}`
- `page.get_images(full=True)` → `doc.extract_image(xref)` 사용
- 파일명: `<stem>_p<N>_img<I>.<ext>`

**평가기준:**
- 보고서 PDF에서 최소 2장 추출 (차트 2개)
- 각 dict에 5개 키 모두 존재
- 저장된 파일이 PIL.Image.open으로 열림


In [ ]:
def extract_embedded_images(pdf_path: str, out_dir: str) -> list[dict]:
    """PDF에 임베드된 이미지를 파일로 분리."""
    # ---- 여기에 코드 작성 ----
    # 1) fitz.open(pdf_path)
    # 2) for page in doc: for xref in page.get_images(full=True):
    # 3) doc.extract_image(xref) → {"image": bytes, "ext": str, ...}
    return []


# 테스트
items = extract_embedded_images(str(PDF_REPORT), str(IMG_DIR))
print(f"추출된 이미지: {len(items)}")
for it in items:
    print(f"  p{it['page']} idx{it['index']}: {Path(it['path']).name} ({it['width']}x{it['height']})")


---
## 문제 3: GPT-4o Vision으로 이미지 캡션 생성

추출한 이미지를 base64로 인코딩하여 Vision LLM에 전달하고 한국어 캡션을 받아오세요.

**요구사항:**
- 함수 시그니처: `caption_image(image_path: str, prompt: str = "이 이미지를 한국어로 1~2문장으로 설명하세요.") -> str`
- `data:image/png;base64,...` URL 형태로 Vision에 전달
- `HumanMessage(content=[{"type":"text",...}, {"type":"image_url","image_url":{"url":...}}])`
- 반환: LLM 응답 텍스트

**평가기준:**
- `caption_image(path)`가 빈 문자열이 아니어야 함
- 캡션 길이 >= 10자


In [ ]:
def caption_image(image_path: str, prompt: str = "이 이미지를 한국어로 1~2문장으로 설명하세요.") -> str:
    """Vision LLM으로 이미지 캡션 생성."""
    # ---- 여기에 코드 작성 ----
    # 1) Path(image_path).read_bytes() → base64.b64encode
    # 2) ext에 따라 mime 결정
    # 3) HumanMessage(content=[text, image_url]) 구성
    # 4) vision.invoke()
    return ""


# 테스트 — 추출한 이미지 중 1장
if items:
    cap = caption_image(items[0]["path"])
    print(f"📝 캡션: {cap}")


---
## 문제 4: 차트 → Pydantic 구조 복원

막대그래프 이미지를 보고 Vision LLM이 카테고리와 값을 JSON으로 복원하도록 하세요. Pydantic으로 검증합니다.

**요구사항:**
- Pydantic 모델: `class ChartData(BaseModel): title: str; categories: list[str]; values: list[float]; unit: str`
- 함수 시그니처: `restore_chart(image_path: str) -> ChartData`
- Vision에 "JSON으로만 답하라" 지시 후 응답을 파싱하여 `ChartData(**parsed)` 반환
- LLM 응답에서 ``` 코드블록은 제거

**평가기준:**
- `categories`와 `values` 길이가 같음
- `len(values) >= 3`
- 모든 `values`가 float


In [ ]:
class ChartData(BaseModel):
    title: str = Field(..., description="차트 제목")
    categories: list[str] = Field(..., description="x축 카테고리")
    values: list[float] = Field(..., description="y축 수치")
    unit: str = Field("", description="수치 단위 (예: 억원)")


def restore_chart(image_path: str) -> ChartData:
    """차트 이미지 → ChartData 구조 복원."""
    # ---- 여기에 코드 작성 ----
    # 1) base64 인코딩 + image_url 메시지
    # 2) prompt: title/categories/values/unit JSON만 출력하도록
    # 3) 응답에서 ``` 제거 후 json.loads
    # 4) ChartData(**parsed)
    return ChartData(title="", categories=[], values=[], unit="")


# 테스트 — 첫 번째 차트 이미지
if items:
    ch = restore_chart(items[0]["path"])
    print(f"📊 {ch.title}")
    for c, v in zip(ch.categories, ch.values):
        print(f"  {c}: {v} {ch.unit}")


---
## 문제 5: PyMuPDF로 표 영역 탐지

PyMuPDF 1.23+ 의 `page.find_tables()` API로 PDF 안의 표 위치를 찾는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `detect_tables(pdf_path: str) -> list[dict]`
- 각 dict: `{"page", "bbox", "rows", "cols"}` — `bbox`는 `(x0, y0, x1, y1)` tuple
- `tables = page.find_tables()` → `tables.tables`

**평가기준:**
- 보고서 PDF에서 최소 1개 이상 표 검출
- bbox가 4-tuple


In [ ]:
def detect_tables(pdf_path: str) -> list[dict]:
    """페이지마다 표 영역(bbox) 탐지."""
    # ---- 여기에 코드 작성 ----
    out = []
    return out


# 테스트
tabs = detect_tables(str(PDF_REPORT))
print(f"검출된 표: {len(tabs)}")
for t in tabs:
    print(f"  p{t['page']} bbox={tuple(round(v,1) for v in t['bbox'])} rows={t['rows']} cols={t['cols']}")


---
## 문제 6: PDF 표를 pandas DataFrame으로 추출

검출된 표를 `pandas.DataFrame`으로 변환하세요.

**요구사항:**
- 함수 시그니처: `extract_tables_as_dataframes(pdf_path: str) -> list[pd.DataFrame]`
- 각 표마다 `tb.to_pandas()` 사용 (PyMuPDF 내장)
- 결과 DataFrame은 빈 컬럼/행을 제거(`dropna(how="all")`)

**평가기준:**
- 보고서 PDF에서 최소 1개 DataFrame 반환
- 첫 DataFrame의 `shape[0] >= 2, shape[1] >= 2`
- 컬럼명이 헤더로 잡혀 있음(중복/None 허용)


In [ ]:
def extract_tables_as_dataframes(pdf_path: str) -> list[pd.DataFrame]:
    """PDF 표를 DataFrame 리스트로."""
    # ---- 여기에 코드 작성 ----
    out = []
    return out


# 테스트
dfs = extract_tables_as_dataframes(str(PDF_REPORT))
print(f"추출 표 수: {len(dfs)}")
for i, df in enumerate(dfs):
    print(f"\n=== Table {i+1} {df.shape} ===")
    print(df.head().to_string(index=False))


---
## 문제 7: 표 정제 — 숫자 컬럼 변환 + 결측 처리

문자열로 들어온 숫자 컬럼(예: `"+35%"`, `"188"`, `"1,820"`)을 float으로 변환하는 함수를 작성하세요.

**요구사항:**
- 함수 시그니처: `clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame`
- 콤마(`,`) 제거, 퍼센트(`%`) 제거, 단위(`억`, `만`) 제거, 음수기호(`-`) 보존
- 값의 70% 이상이 숫자로 변환 가능한 컬럼만 float으로 변환
- 변환 불가 값은 `NaN`으로

**평가기준:**
- `"1,820" → 1820.0`, `"+35%" → 35.0`
- 문자열만 있는 컬럼은 그대로 유지
- 원본 df는 변경되지 않음(불변성)


In [ ]:
import re

def clean_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """문자열 숫자 컬럼을 float으로."""
    # ---- 여기에 코드 작성 ----
    # 1) df.copy()로 시작
    # 2) 각 컬럼마다 콤마/퍼센트/단위 제거 후 float 변환 시도
    # 3) 변환 성공률 70% 이상이면 컬럼 교체
    return df.copy()


# 테스트
if dfs:
    cleaned = clean_numeric_columns(dfs[0])
    print(cleaned.dtypes)
    print(cleaned.head())


---
## 문제 8: 표 → Markdown (LLM이 읽기 좋은 형식)

DataFrame을 Markdown 표 문자열로 변환하세요. RAG 컨텍스트에 표를 자연어처럼 끼워넣기 위해 사용합니다.

**요구사항:**
- 함수 시그니처: `df_to_markdown(df: pd.DataFrame, caption: str = "") -> str`
- 첫 줄에 `**caption**`(주어진 경우)
- `df.to_markdown(index=False)` 사용 (없으면 직접 구현 OK — `|` 구분, 헤더 아래 구분선)
- 결과는 `str` 타입

**평가기준:**
- 결과에 `|` 문자가 헤더+구분선 포함 최소 2회 등장
- `caption` 주어지면 `**caption**` 포함


In [ ]:
def df_to_markdown(df: pd.DataFrame, caption: str = "") -> str:
    """DataFrame을 Markdown 표로."""
    # ---- 여기에 코드 작성 ----
    return ""


# 테스트
if dfs:
    md_text = df_to_markdown(cleaned, caption="2025 Q4 사업부별 실적")
    print(md_text)


---
## 문제 9: Vision OCR — 페이지 이미지에서 텍스트 추출

`render_pages_to_png`로 만든 페이지 이미지를 GPT-4o Vision에 보내 텍스트를 OCR하세요.

**요구사항:**
- 함수 시그니처: `vision_ocr(image_path: str) -> str`
- 프롬프트: "이 이미지의 모든 텍스트를 원본 순서대로 추출하세요. 표는 행/열 구조를 보존하세요. 해석/요약 금지."
- 반환: OCR 텍스트 (한국어 보존)

**평가기준:**
- 첫 페이지 OCR 결과에 "Modu Tech" 또는 "분기" 류 키워드 포함
- 빈 문자열 아님


In [ ]:
def vision_ocr(image_path: str) -> str:
    """페이지 이미지 → 원본 텍스트 OCR."""
    # ---- 여기에 코드 작성 ----
    return ""


# 테스트
ocr_text = vision_ocr(paths[0])
print(f"OCR 결과(앞 300자):\n{ocr_text[:300]}")


---
## 문제 10: Whisper로 오디오 파일 → 텍스트 추출

LangChain의 `OpenAIWhisperParser`로 회의 mp3를 텍스트화하세요.

**요구사항:**
- 함수 시그니처: `transcribe_audio(audio_path: str) -> dict`
- LangChain 패턴: `OpenAIWhisperParser().lazy_parse(Blob.from_path(path))` → `list[Document]`
- 반환: `{"source": str, "text": str, "n_chunks": int, "duration_sec": float}`
- `duration_sec`은 `cv2.VideoCapture(path)` 또는 `mutagen` 없이도 OK — 안 되면 0.0
- 25MB 넘는 파일은 자동으로 여러 chunk로 잘림 → text 합쳐서 반환

**평가기준:**
- `text` 길이 >= 30자 (한국어 회의 발화)
- `source`는 파일명만 (경로 X)
- 4개 키 모두 존재


In [ ]:
from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser
from langchain_core.document_loaders.blob_loaders import Blob


def transcribe_audio(audio_path: str) -> dict:
    """Whisper로 오디오 → 텍스트 + 메타."""
    # ---- 여기에 코드 작성 ----
    # 1) OpenAIWhisperParser()
    # 2) Blob.from_path(audio_path)
    # 3) list(parser.lazy_parse(blob))
    # 4) page_content 들을 join
    # 5) duration은 cv2.VideoCapture로 (mp3도 동작) — 실패 시 0.0
    return {"source": "", "text": "", "n_chunks": 0, "duration_sec": 0.0}


# 테스트
sample_audio = audio_files[0]
result = transcribe_audio(str(sample_audio))
print(f"📁 {result['source']}  ({result['duration_sec']:.1f}s, {result['n_chunks']} chunk)")
print(f"📝 {result['text'][:200]}...")


---
## 문제 11: 오디오 transcript → 청크 Document 리스트

문제 10에서 받은 transcript를 RAG가 검색하기 좋게 청크로 나누세요. `RecursiveCharacterTextSplitter` 사용.

**요구사항:**
- 함수 시그니처: `chunk_audio_transcript(transcript: dict, chunk_size: int = 300, chunk_overlap: int = 50) -> list[Document]`
- `LangChain`의 `RecursiveCharacterTextSplitter`로 `transcript["text"]` 분할
- 각 Document `metadata`: `{"source", "element_type": "audio_chunk", "chunk_index"}`
- `source`는 transcript에서 그대로 가져오기
- 빈 청크는 제외

**평가기준:**
- 반환 길이 >= 1
- 모든 Document에 3개 메타 키 존재
- `chunk_index`가 0부터 1씩 증가
- 모든 `page_content`가 빈 문자열 아님


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_audio_transcript(transcript: dict, chunk_size: int = 300, chunk_overlap: int = 50) -> list[Document]:
    """transcribe_audio 결과를 청크 Document로 분할."""
    # ---- 여기에 코드 작성 ----
    # 1) splitter = RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=...)
    # 2) chunks = splitter.split_text(transcript["text"])
    # 3) Document(page_content=c, metadata={...})
    return []


# 테스트
audio_docs = chunk_audio_transcript(result, chunk_size=200, chunk_overlap=40)
print(f"오디오 청크: {len(audio_docs)}개")
for d in audio_docs[:3]:
    print(f"  [{d.metadata['element_type']}] #{d.metadata['chunk_index']}  {d.page_content[:80]!r}")


---
## 문제 12: 비디오 프레임 샘플링 + Vision 캡션 → Document

비디오에서 매 N초마다 프레임을 뽑아 GPT-4o Vision으로 캡션을 만들고 Document로 묶으세요.

**요구사항:**
- 함수 시그니처: `caption_video_frames(video_path: str, out_dir: str, every_sec: int = 10) -> list[Document]`
- `cv2.VideoCapture` + `cap.set(cv2.CAP_PROP_POS_MSEC, sec*1000)` + `cap.read()`로 시각 점프 후 frame 추출
- 각 frame을 `<out_dir>/<stem>_t<sec>s.png`로 저장
- `caption_image()` (문제 3에서 만든 함수) 재사용해 캡션 생성
- Document `metadata`: `{"source", "element_type": "video_frame_caption", "timestamp_sec"}`
- 마지막 프레임까지 자연스럽게 끝나야 함 (`read()`가 False면 종료)

**평가기준:**
- Document 1개 이상 (60초 영상 × every_sec=10 → ~6개 기대)
- 모든 `page_content`가 빈 문자열 아님 (캡션 생성됨)
- `timestamp_sec`이 정수, 오름차순
- 저장된 png 파일이 실제로 존재


In [ ]:
def caption_video_frames(video_path: str, out_dir: str, every_sec: int = 10) -> list[Document]:
    """비디오에서 매 every_sec마다 frame 추출 + Vision 캡션 → Document 리스트."""
    # ---- 여기에 코드 작성 ----
    # 1) out_dir mkdir
    # 2) cv2.VideoCapture
    # 3) sec=0부터 every_sec씩 증가하며 cap.set+read
    # 4) cv2.imwrite로 frame 저장
    # 5) caption_image(path)로 캡션
    # 6) Document(page_content=caption, metadata={...})
    return []


# 테스트
video_docs = caption_video_frames(str(VIDEO_PATH), str(FRAMES_DIR), every_sec=15)
print(f"비디오 프레임 Document: {len(video_docs)}개")
for d in video_docs:
    print(f"  t={d.metadata['timestamp_sec']:>3}s  {d.page_content[:80]}")


---
## 문제 13: 비디오의 오디오 트랙 분리 + Whisper transcribe

비디오에서 오디오 트랙만 추출(`ffmpeg`)한 뒤 문제 10의 `transcribe_audio`를 재사용하세요.

**요구사항:**
- 함수 시그니처: `transcribe_video_audio(video_path: str, audio_out_dir: str) -> dict`
- 출력 mp3: `<audio_out_dir>/<stem>.mp3`
- ffmpeg 명령: `ffmpeg -y -i <video> -vn -acodec libmp3lame <out.mp3>` (또는 `-acodec copy` 후 컨테이너만 변경 — 영상에 mp3 트랙이 있다면)
- `subprocess.run(..., check=True, capture_output=True)` 권장
- 추출된 mp3를 `transcribe_audio()`에 넣어 결과 반환
- 반환 dict에 `"video_source": <video filename>` 키 추가

**평가기준:**
- 반환 dict에 `text`, `source`, `video_source` 키 존재
- `text` 길이 >= 30자
- mp3 파일이 디스크에 실제로 생성됨

> 💡 **시스템 ffmpeg 확인**: `!ffmpeg -version` 으로 사전 체크.


In [ ]:
def transcribe_video_audio(video_path: str, audio_out_dir: str) -> dict:
    """비디오 → 오디오 트랙 분리 → Whisper transcribe."""
    # ---- 여기에 코드 작성 ----
    # 1) mkdir
    # 2) audio_path = <audio_out_dir>/<stem>.mp3
    # 3) subprocess.run(["ffmpeg", "-y", "-i", video_path, "-vn", "-acodec", "libmp3lame", str(audio_path)],
    #                    check=True, capture_output=True)
    # 4) result = transcribe_audio(str(audio_path))
    # 5) result["video_source"] = Path(video_path).name
    return {}


# 테스트
va_result = transcribe_video_audio(str(VIDEO_PATH), str(AV_TMP_DIR))
print(f"🎬 video: {va_result['video_source']}")
print(f"🎙️  extracted: {va_result['source']}  ({va_result.get('duration_sec', 0):.1f}s)")
print(f"📝 {va_result['text'][:200]}...")


---
## 문제 14: PDF + 오디오 + 비디오를 하나의 `Document` 리스트로 통합 🏁

문제 1~13의 결과를 단일 `Document` 리스트로 묶는 빌더를 작성하세요. Weekend 3 멀티모달 RAG의 직접 입력이 됩니다.

**요구사항:**
- 함수 시그니처:
  ```python
  build_multimodal_documents(
      pdf_path: str,
      img_dir: str,
      audio_paths: list[str] | None = None,
      video_path: str | None = None,
      av_tmp_dir: str | None = None,
  ) -> list[Document]
  ```
- 각 Document의 `metadata`는 모달리티에 따라 다름:
  - 표: `{"source", "page_number", "element_type": "table"}`
  - 이미지: `{"source", "page_number", "element_type": "image_caption"}`
  - 페이지 OCR: `{"source", "page_number", "element_type": "page_ocr"}`
  - 오디오 청크: `{"source", "element_type": "audio_chunk", "chunk_index"}`
  - 비디오 프레임: `{"source", "element_type": "video_frame_caption", "timestamp_sec"}`
  - 비디오 오디오 청크: `{"source", "element_type": "video_audio_chunk", "chunk_index"}`
- 속도용: PDF는 첫 2장 이미지 + 첫 1페이지 OCR만, 비디오는 `every_sec=15`로 OK
- `audio_paths=None` / `video_path=None`이면 해당 모달리티 스킵 (PDF 만 처리 가능)

**평가기준:**
- `pdf_path` + `audio_paths` + `video_path` 모두 주어졌을 때:
  - 반환 길이 >= 6
  - `set(d.metadata["element_type"] for d in docs)` ⊇ {`"table"`, `"image_caption"`, `"page_ocr"`, `"audio_chunk"`, `"video_frame_caption"`, `"video_audio_chunk"`} (모두)
  - 모든 Document에 `source`, `element_type` 키 존재
- `audio_paths=None, video_path=None`일 때: 문제 10(이전 버전)과 동일하게 PDF만 처리


In [ ]:
def build_multimodal_documents(
    pdf_path: str,
    img_dir: str,
    audio_paths: list | None = None,
    video_path: str | None = None,
    av_tmp_dir: str | None = None,
) -> list[Document]:
    """PDF + 오디오 + 비디오 → 단일 Document 리스트."""
    docs: list[Document] = []
    # ---- 여기에 코드 작성 ----
    # 1) PDF 표 (extract_tables_as_dataframes → clean → df_to_markdown)
    # 2) PDF 이미지 (extract_embedded_images → caption_image, 첫 2장)
    # 3) PDF 페이지 OCR (render_pages_to_png + vision_ocr, 첫 페이지)
    # 4) audio_paths: 각 mp3마다 transcribe_audio → chunk_audio_transcript → docs 확장
    # 5) video_path:
    #     - caption_video_frames(every_sec=15) → docs 확장
    #     - transcribe_video_audio → chunk_audio_transcript(element_type 수동 교체: "video_audio_chunk")
    return docs


# 테스트
mm_docs = build_multimodal_documents(
    pdf_path=str(PDF_REPORT),
    img_dir=str(IMG_DIR),
    audio_paths=[str(audio_files[0])],          # 회의 mp3 1개만
    video_path=str(VIDEO_PATH),
    av_tmp_dir=str(AV_TMP_DIR),
)
print(f"총 멀티모달 Documents: {len(mm_docs)}개")
print(f"\n모달리티별 분포:")
counter = Counter(d.metadata["element_type"] for d in mm_docs)
for k, v in counter.most_common():
    print(f"  {k:<22} {v}")


---
## 문제 15: 멀티모달 Document를 단일 FAISS 인덱스에 통합

문제 14에서 만든 `Document` 리스트를 통째로 FAISS에 임베딩·인덱싱하세요. 표·이미지·오디오·비디오가 *같은 벡터 공간*에 들어갑니다.

**요구사항:**
- 함수 시그니처: `build_faiss_index(docs: list[Document]) -> FAISS`
- `OpenAIEmbeddings(model="text-embedding-3-small")` (이미 `embeddings` 객체로 셋업됨)
- `FAISS.from_documents(docs, embeddings)` 패턴
- `metadata`는 자동 보존됨 (`element_type` 검색 시 활용)
- 빈 docs 입력 시 `ValueError` raise

**평가기준:**
- 반환 객체가 `FAISS` 타입
- `vs.similarity_search("매출")` 호출 시 `Document` 리스트 반환
- 반환된 Document들이 원본 `metadata["element_type"]`을 유지


In [ ]:
def build_faiss_index(docs: list[Document]) -> FAISS:
    """멀티모달 Document 리스트 → FAISS 인덱스."""
    # ---- 여기에 코드 작성 ----
    # 1) 빈 입력 체크
    # 2) FAISS.from_documents(docs, embeddings)
    return None


# 테스트
vs = build_faiss_index(mm_docs)
hits = vs.similarity_search("Modu Tech 매출", k=4)
print(f"검색 결과 {len(hits)}개:")
for h in hits:
    et = h.metadata.get("element_type", "?")
    print(f"  [{et:<22}] {h.page_content[:70]}")


---
## 문제 16: `element_type` 별 가중치 검색

같은 유사도라도 표는 더 신뢰하고 이미지 캡션은 덜 신뢰하고 싶을 때. `similarity_search_with_score`로 점수 받아 가중치 곱해 재정렬.

**요구사항:**
- 함수 시그니처:
  ```python
  weighted_search(vs: FAISS, query: str,
                  weights: dict[str, float] | None = None,
                  k: int = 5, fetch_k: int = 20) -> list[tuple[Document, float]]
  ```
- 기본 `weights`:
  ```python
  {"table": 1.4, "page_ocr": 1.0, "image_caption": 0.8,
   "audio_chunk": 1.1, "video_frame_caption": 0.9, "video_audio_chunk": 1.1}
  ```
- 동작:
  1. `similarity_search_with_score(query, k=fetch_k)` — FAISS는 *거리*(낮을수록 좋음) 반환
  2. `score = weights.get(element_type, 1.0) / (distance + 1e-9)` — *높을수록 좋음*으로 변환 + 가중
  3. score 내림차순 정렬 → 상위 k개 반환

**평가기준:**
- 반환이 `[(Document, float)]` 리스트
- 가중치 0 주면 그 element_type은 결과에서 사실상 사라짐
- 기본 가중치 사용 시 표(table)가 같은 거리의 image_caption보다 위에 옴


In [ ]:
def weighted_search(vs: FAISS, query: str,
                    weights: dict | None = None,
                    k: int = 5, fetch_k: int = 20) -> list[tuple]:
    """element_type 가중치를 곱한 재정렬 검색."""
    # ---- 여기에 코드 작성 ----
    # 1) default weights
    # 2) hits = vs.similarity_search_with_score(query, k=fetch_k)
    # 3) (doc, weight / (distance + 1e-9)) 계산
    # 4) score desc 정렬 → 상위 k
    return []


# 테스트
print("🔍 'Q4 매출 차트' — 기본 가중치 (표 우대)")
for doc, sc in weighted_search(vs, "Q4 매출 차트", k=5):
    et = doc.metadata.get("element_type", "?")
    print(f"  {sc:.3f}  [{et:<22}] {doc.page_content[:60]}")

print("\n🔍 같은 쿼리 — 이미지 caption만 (다른 모달 0 가중치)")
zeros = {"table": 0, "page_ocr": 0, "audio_chunk": 0, "video_audio_chunk": 0}
for doc, sc in weighted_search(vs, "Q4 매출 차트", weights=zeros, k=3):
    et = doc.metadata.get("element_type", "?")
    print(f"  {sc:.3f}  [{et:<22}] {doc.page_content[:60]}")


---
## 문제 17: 멀티모달 컨텍스트로 RAG 답변 생성

가중치 검색 결과를 LLM 프롬프트에 *모달리티 별로 묶어* 넣고 답변을 만드세요. 모달리티별로 컨텍스트가 어떻게 다른지 LLM이 인지하도록.

**요구사항:**
- 함수 시그니처:
  ```python
  multimodal_rag_answer(vs: FAISS, query: str, k: int = 6) -> dict
  ```
- 동작:
  1. `weighted_search(vs, query, k=k)`로 컨텍스트 수집
  2. 컨텍스트를 `element_type`별로 그룹핑해서 프롬프트 섹션으로 정리
     - 예: `### 표 (PDF)\n<content>` / `### 이미지 캡션\n<content>` / `### 회의 발화\n<content>` / `### 비디오 프레임\n<content>`
  3. 시스템 메시지: "각 섹션은 다른 모달리티에서 추출되었음. 표는 정량 정보, 발화는 발언 내용, 프레임은 시각 정보."
  4. `llm.invoke([SystemMessage, HumanMessage])`로 답변 생성
- 반환: `{"answer": str, "sources": list[dict]}` — `sources`는 각 컨텍스트의 `{"element_type", "snippet": page_content[:80]}`

**평가기준:**
- 반환 dict에 `answer`, `sources` 키 존재
- `len(sources) <= k`
- `answer` 길이 >= 20자


In [ ]:
ELEMENT_LABELS = {
    "table":               "표 (PDF)",
    "page_ocr":            "페이지 텍스트 (PDF OCR)",
    "image_caption":       "이미지 캡션",
    "audio_chunk":         "회의 발화",
    "video_frame_caption": "비디오 프레임",
    "video_audio_chunk":   "비디오 발화",
}


def multimodal_rag_answer(vs: FAISS, query: str, k: int = 6) -> dict:
    """멀티모달 컨텍스트로 RAG 답변 + 출처 dict 반환."""
    # ---- 여기에 코드 작성 ----
    # 1) hits = weighted_search(vs, query, k=k)
    # 2) element_type별 그룹핑
    # 3) 섹션 문자열 구성 → 시스템/유저 메시지
    # 4) llm.invoke
    return {"answer": "", "sources": []}


# 테스트
r = multimodal_rag_answer(vs, "Modu Tech의 Q4 매출과 회의에서 언급된 주요 전략은?", k=6)
print("=" * 70)
print(r["answer"])
print("=" * 70)
print(f"\n출처 {len(r['sources'])}개:")
for s in r["sources"]:
    print(f"  [{s['element_type']:<22}] {s['snippet']!r}")


---
## 문제 18: LLM-as-Judge — 답변 충실도/관련성 평가 🏁

문제 17의 답변이 *주어진 컨텍스트에 충실한지* 다른 LLM으로 평가하세요. RAG 운영 시 hallucination 모니터링의 기본 패턴.

**요구사항:**
- 함수 시그니처:
  ```python
  judge_answer(query: str, answer: str, sources: list[dict]) -> dict
  ```
- Pydantic 스키마:
  ```python
  class JudgeVerdict(BaseModel):
      faithfulness: int = Field(..., ge=1, le=5, description="컨텍스트만 보고 정당화 가능한가")
      relevance:    int = Field(..., ge=1, le=5, description="질문에 실제로 답하는가")
      reasoning:    str = Field(..., description="짧은 근거(한국어)")
  ```
- 동작:
  1. `llm.with_structured_output(JudgeVerdict)`로 구조화 출력 강제
  2. 시스템: "당신은 RAG 답변 평가자. faithfulness=5는 컨텍스트로 완전히 정당화. relevance=5는 질문에 직접 답."
  3. 유저: 질문/답변/sources(각 element_type + snippet)을 보여줌
  4. 반환: `verdict.model_dump()`

**평가기준:**
- 반환 dict에 `faithfulness`, `relevance`, `reasoning` 3개 키
- 정수 범위 1~5
- `reasoning`이 빈 문자열 아님

**전체 파이프라인 통합 — 마지막 검증**:
```
PDF/Audio/Video → Document (#14) → FAISS (#15)
              → 가중치 검색 (#16) → 멀티모달 답변 (#17) → Judge 평가 (#18)
```

In [ ]:
class JudgeVerdict(BaseModel):
    faithfulness: int = Field(..., ge=1, le=5, description="컨텍스트만 보고 정당화 가능한가")
    relevance:    int = Field(..., ge=1, le=5, description="질문에 실제로 답하는가")
    reasoning:    str = Field(..., description="짧은 근거(한국어)")


def judge_answer(query: str, answer: str, sources: list[dict]) -> dict:
    """LLM-as-Judge: 답변 충실도/관련성 평가."""
    # ---- 여기에 코드 작성 ----
    # 1) judge_llm = llm.with_structured_output(JudgeVerdict)
    # 2) sources를 텍스트로 직렬화
    # 3) sys/user 메시지
    # 4) judge_llm.invoke(...).model_dump()
    return {"faithfulness": 0, "relevance": 0, "reasoning": ""}


# 전체 파이프라인 한번에
query = "Modu Tech의 Q4 매출과 회의에서 언급된 주요 전략은?"
r = multimodal_rag_answer(vs, query, k=6)
verdict = judge_answer(query, r["answer"], r["sources"])

print("=" * 70)
print(f"질문: {query}")
print(f"답변: {r['answer']}")
print("=" * 70)
print(f"⚖️  Faithfulness: {verdict['faithfulness']}/5")
print(f"⚖️  Relevance:    {verdict['relevance']}/5")
print(f"💭 Reasoning:    {verdict['reasoning']}")
